In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# مسار البيانات المعالجة
PROCESSED_DIR = Path("../data/processed/part_0001.parquet")

# 1. قراءة البيانات (Parquet يقرأ المجلد كاملاً كأنه ملف واحد)
print("📂 Loading Processed Data...")
try:
    df = pd.read_parquet(PROCESSED_DIR)
    print(f"✅ Loaded successfully! Shape: {df.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    # توقف هنا إذا فشل التحميل

# 2. عرض عينة عشوائية
print("\n🔍 Random Sample:")
display(df)

# 3. فحص أنواع البيانات (Data Types)
print("\n📊 Data Types:")
print(df.dtypes)

# 4. فحص عمود القوائم (Diagnoses List) - أهم اختبار!
# نتأكد أنها List حقيقية وليست نصاً
print("\n🧪 Inspecting Diagnoses Lists:")
sample_diag = df['diagnosis_names'].iloc[0]
print(f"Type of cell content: {type(sample_diag)}")
print(f"Content example: {sample_diag}")

if isinstance(sample_diag, np.ndarray) or isinstance(sample_diag, list):
    print("✅ GREAT! Diagnoses are stored as Arrays/Lists.")
else:
    print("⚠️ WARNING: Diagnoses might be stored as Strings!")

# 5. فحص القيم المفقودة (Nulls)
print("\n🕳️ Missing Values Check:")
print(df.isnull().sum())

# 6. فحص المنطق (Business Logic Check)
# هل هناك مريض لديه وصفة دواء ولكن لا يوجد له عمر أو وزن؟ (مشكلة في الـ Join)
orphan_records = df[df['anchor_age'].isnull()]

# 7. فحص التزامن بين الأكواد والأسماء (Alignment Check)
print("\n⚖️ Checking Alignment (Codes vs Names lengths)...")

# دالة لحساب الفرق في الطول
def check_length_mismatch(row):
    return len(row['diagnosis_codes']) != len(row['diagnosis_names'])

# تطبيق الفحص
mismatches = df[df.apply(check_length_mismatch, axis=1)]

if len(mismatches) > 0:
    print(f"⚠️ CRITICAL WARNING: Found {len(mismatches)} rows where Codes count != Names count.")
    display(mismatches[['hadm_id', 'diagnosis_codes', 'diagnosis_names']].head(3))
else:
    print("✅ PERFECT ALIGNMENT: Every diagnosis code has a corresponding name.")
    
if len(orphan_records) > 0:
    print(f"\n⚠️ Alert: Found {len(orphan_records)} prescriptions with missing patient info (Join failed for some IDs).")
else:
    print("\n✅ Integrity Check Passed: All prescriptions link to a patient.")

📂 Loading Processed Data...
❌ Error loading data: [Errno 2] No such file or directory: '..\\data\\processed\\part_0001.parquet'

🔍 Random Sample:


,subject_id,hadm_id,starttime,drug,ndc,dose_val_rx,dose_unit_rx,gender,anchor_age,weight,diagnosis_codes,diagnosis_names
0,10000032,22595853,2180-05-08 08:00:00,Furosemide,5.107901e+10,40,mg,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
1,10000032,22595853,2180-05-07 02:00:00,Ipratropium Bromide Neb,4.879801e+08,1,NEB,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
2,10000032,22595853,2180-05-07 01:00:00,Furosemide,5.107901e+10,20,mg,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
3,10000032,22595853,2180-05-07 01:00:00,Potassium Chloride,2.450041e+08,40,mEq,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
4,10000032,22595853,2180-05-07 00:00:00,Sodium Chloride 0.9% Flush,0.000000e+00,3,mL,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,10049746,24332085,2136-11-26 12:00:00,CycloPHOSPHAMIDE,1.001910e+10,680,mg,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."
99996,10049746,24332085,2137-01-01 08:00:00,Ondansetron,5.107905e+10,16,mg,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."
99997,10049746,24332085,2136-11-26 20:00:00,OLANZapine (Disintegrating Tablet),5.026806e+10,2.5,mg,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."
99998,10049746,24332085,2136-11-27 21:00:00,D5W,0.000000e+00,300,mL,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."



📊 Data Types:
subject_id                  Int64
hadm_id                     Int64
starttime          datetime64[ns]
drug                       object
ndc                       float64
dose_val_rx                object
dose_unit_rx               object
gender                   category
anchor_age                  int64
weight                    float64
diagnosis_codes            object
diagnosis_names            object
dtype: object

🧪 Inspecting Diagnoses Lists:
Type of cell content: <class 'numpy.ndarray'>
Content example: ['Portal hypertension' 'Other ascites'
 'Cirrhosis of liver without mention of alcohol'
 'Unspecified viral hepatitis C without hepatic coma'
 'Chronic airway obstruction, not elsewhere classified'
 'Bipolar disorder, unspecified' 'Posttraumatic stress disorder'
 'Personal history of tobacco use']
✅ GREAT! Diagnoses are stored as Arrays/Lists.

🕳️ Missing Values Check:
subject_id             0
hadm_id                0
starttime            102
drug                  

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. تحديد مسار الملف الذهبي الجديد
GOLD_FILE = Path("../data/processed/GOLD_patient_records.parquet")

print(f"📂 Loading GOLD Dataset from: {GOLD_FILE.name} ...")

try:
    df = pd.read_parquet(GOLD_FILE)
    print(f"✅ Loaded successfully! Shape: {df.shape}")
    print(f"   (Expectation: Rows = Number of unique patients)")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    # توقف هنا
    raise e

# 2. عرض عينة لفهم الهيكلية الجديدة
print("\n🔍 Random Patient Profile:")
# نوسع العرض لنرى القوائم كاملة
pd.set_option('display.max_colwidth', 100) 
display(df.sample(10))

# 3. اختبار "مريض واحد = صف واحد" (Uniqueness Check)
print("\n🆔 Checking Patient Uniqueness...")
if df['subject_id'].is_unique:
    print(f"✅ PASSED: All {len(df)} rows represent unique patients.")
else:
    print(f"⚠️ FAILED: Duplicates found! ({df['subject_id'].duplicated().sum()} duplicates)")

# 4. فحص القوائم (Diagnosis Names & Medication List)
print("\n🧪 Inspecting Lists Structure:")

for col in ['diagnosis_names', 'medication_list']:
    print(f"   Checking column: {col}...")
    if col not in df.columns:
        print(f"   ❌ Missing column {col}!")
        continue
        
    sample_val = df[col].iloc[0]
    print(f"      -> Type: {type(sample_val)}")
    print(f"      -> Sample Content: {sample_val[:10]} ... (Truncated)")
    
    if isinstance(sample_val, (list, np.ndarray)):
        # فحص هل القائمة فارغة؟
        empty_count = df[col].apply(lambda x: len(x) == 0).sum()
        print(f"      ✅ Valid List Format. (Empty lists count: {empty_count})")
    else:
        print("      ⚠️ WARNING: Not a list!")

# 5. فحص القيم المفقودة (Nulls)
print("\n🕳️ Missing Values Check:")
print(df.isnull().sum())

# 6. فحص منطقي (هل يوجد دواء بدون تشخيص؟)
# هذا ليس خطأ بالضرورة، لكنه مفيد للمودل
print("\n🧠 Logic Check:")
no_meds = df[df['medication_list'].apply(len) == 0]
no_diag = df[df['diagnosis_names'].apply(len) == 0]

print(f"   - Patients with NO medications: {len(no_meds)}")
print(f"   - Patients with NO diagnoses: {len(no_diag)}")

if len(no_meds) == 0 and len(no_diag) == 0:
    print("✅ Excellent! Every patient has at least one med and one diagnosis.")

📂 Loading GOLD Dataset from: GOLD_patient_records.parquet ...
✅ Loaded successfully! Shape: (196738, 9)
   (Expectation: Rows = Number of unique patients)

🔍 Random Patient Profile:


,subject_id,hadm_id,gender,anchor_age,weight,medication_list,dosage_list,unit_list,diagnosis_names
130517,16659685,23030610,M,23,NaN,"[0.45% Sodium Chloride, 0.9% Sodium Chloride, Simvastatin, NS, MetRONIDAZOLE (FLagyl), 0.45% Sod...","[1000, 1000, 20, 100, 500, 1000, 100, 4, 500, 200, 400, 3, 1000, 4, 0.5]","[mL, mL, mg, mL, mg, mL, mL, gm, mg, mL, mg, mL, mL, mg, mL]","[Syncope and collapse, Thrombocytopenia, unspecified, Intestinal infection due to other organism..."
97306,14972476,28430535,F,59,167.144444,"[Pantoprazole, Clonazepam, Heparin, Clonazepam, Sodium Chloride 0.9% Flush, Levothyroxine Sodiu...","[40, 1, 5000, 1, 3, 112, 600, 250, 40, 1000, 1-2, 0.25, 15, 100, 20, 500, 30, 2]","[mg, mg, UNIT, mg, mL, mcg, mg, mg, mg, mL, TAB, mg, mg, mg, mg, mg, mg, mg]","[Benign neoplasm of ovary, Polyp of corpus uteri, Unspecified acquired hypothyroidism, Unspecifi..."
139342,17106592,27564537,M,66,146.025000,"[Acetaminophen, OxyCODONE (Immediate Release), Lactated Ringers, Ciprofloxacin HCl, Influenza Va...","[650, 5, 1000, 500, 0.5, 3-10, 50, 400, 1, 2, 200, 1000, 100, 500, 1000, 8.6, 1000, 1, 3, 40, 10...","[mg, mg, mL, mg, mL, mL, mg, mg, VIAL, g, mL, mg, mL, mg, mL, mg, mL, VIAL, g, mg, mg, mg, g, mL...","[Abscess of liver, Klebsiella pneumoniae [K. pneumoniae] as the cause of diseases classified els..."
37956,11943894,29328542,M,59,329.250000,"[Indomethacin, Diazepam, Aluminum-Magnesium Hydrox.-Simethicone, Gabapentin, Ondansetron ODT, Di...","[50, 5, 30, 300, 4, 10, 10, 6, 2, 10, 30, 40, 50, 300, 100, 1, 600, 650, 5, 50, 1]","[mg, mg, mL, mg, mg, mg, mg, mg, mg, mg, mL, mg, mg, mg, mg, Appl, mg, mg, mg, mg, mg]","[Depressive disorder, not elsewhere classified, Cocaine dependence, continuous, Chronic pancreat..."
186700,19487571,20562493,M,76,202.000000,"[Senna, Potassium Chloride, Losartan Potassium, Warfarin, Atorvastatin, Eplerenone, Warfarin, PN...","[8.6, 40, 50, 3, 80, 25, 1, 0.5, 1, 40, 325-650, 60, 0.125, 1000, 100, 0.5, 125, 0.4, 60, 5000, ...","[mg, mEq, mg, mg, mg, mg, dose, mL, mg, mg, mg, mEq, mg, mL, mg, mL, mcg, mg, mEq, UNIT, mg, mg,...","[Ventricular tachycardia, Chronic systolic (congestive) heart failure, Unspecified atrial flutte..."
49027,12507121,22610205,F,43,NaN,"[Potassium Chl 20 mEq / 1000 mL D5 1/2 NS, Sodium Chloride 0.9% Flush, DiphenhydrAMINE, HYDROmo...","[1000, 3, 50, 0.25, 1000, 0.5, 200, 400, 0.5, 100, 500, 1000, 20, 4, 5000]","[mL, mL, mg, mg, mg, mL, mL, mg, mL, mL, mg, mL, mg, mg, UNIT]","[Urinary tract infection, site not specified, Esophageal reflux, Asthma, unspecified type, unspe..."
135283,16901518,28918394,M,82,NaN,"[5% Dextrose, Heparin Sodium, Cholestyramine, Clopidogrel, Albuterol-Ipratropium, Morphine Sulfa...","[250, 25,000, 4, 75, 1-2, 2-4, 0.5, 3, 2-4, 325, 12.5, 6.25, 12.5, 1, 1, 75, 10, 20, 325, 6.25, ...","[mL, UNIT, gm, mg, PUFF, mg, mL, mL, mg, mg, mg, mg, gm, CAP, INH, mg, mg, mg, mg, mg, mg, mL, m...","[Subendocardial infarction, initial episode of care, Unspecified essential hypertension, Pure hy..."
116436,15944096,29867421,F,69,194.540000,"[LOPERamide, Sodium Chloride 0.9% Flush, Enoxaparin Sodium, Sertraline, Levothyroxine Sodium, G...","[2, 3, 30, 100, 50, 1, 1, 50-100, 8.6, 1000, 40, 4, 5-10, 1000, 150, 0, 1000, 50, 2, 100, 1000, ...","[mg, mL, mg, mg, mcg, mg, mg, mg, mg, mL, mg, mg, mg, mL, mg, UNIT, mg, mL, g, mg, mg, mg, mg, m...","[Unilateral primary osteoarthritis, right knee, Cardiomyopathy, unspecified, Unspecified atrial ..."
59972,13061918,22528033,F,26,141.700000,"[Acetaminophen, Metoprolol Tartrate, amLODIPine, Bag, Magnesium Sulfate, Bag, Magnesium Sulfate,...","[650, 6.25, 5, 1, 2, 1, 2, 1500, 5, 1000, 1000, 3-10, 1000, 5, 1000, 6.25, 2.5, 1000, 40, 40, 40...","[mg, mg, mg, BAG, gm, BAG, gm, mg, mg, mL, mg, mL, mL, mg, mL, mg, mg, mg, mg, mg, mEq, mL, mg, ...","[Viral meningitis, unspecified, Rhabdomyolysis, Takotsubo syndrome, Other reaction to spinal and..."
189128,19610569,21883209,M,33,206.200000,"[Potassium Chloride, Spironolactone, Prochlorp


🆔 Checking Patient Uniqueness...
✅ PASSED: All 196738 rows represent unique patients.

🧪 Inspecting Lists Structure:
   Checking column: diagnosis_names...
      -> Type: <class 'numpy.ndarray'>
      -> Sample Content: ['Chronic hepatitis C without mention of hepatic coma' 'Other ascites'
 'Other dependence on machines, supplemental oxygen'
 'Cirrhosis of liver without mention of alcohol' 'Hyperpotassemia'
 'Hyposmolality and/or hyponatremia'
 'Chronic airway obstruction, not elsewhere classified'
 'Asymptomatic human immunodeficiency virus [HIV] infection status'
 'Tobacco use disorder' 'Diarrhea'] ... (Truncated)
      ✅ Valid List Format. (Empty lists count: 134)
   Checking column: medication_list...
      -> Type: <class 'numpy.ndarray'>
      -> Sample Content: ['Sodium Chloride 0.9%  Flush' '0.9% Sodium Chloride' 'Calcium Gluconate'
 'Furosemide' 'Albuterol Inhaler' 'Sodium Polystyrene Sulfonate'
 'Zolpidem Tartrate' 'Emtricitabine-Tenofovir (Truvada)' 'Rifaximin'
 'Sodium Pol

In [12]:
import re

def extract_preadmission_medications_v8(clinical_note):
    # 1. تحديد القسم
    section_pattern = r"(?i)(?:Medications|___)\s+on\s+Admission:(.*?)(?=Discharge\s+Medications:)"
    match = re.search(section_pattern, clinical_note, re.DOTALL)
    
    if not match:
        return []

    meds_section_text = match.group(1).strip()
    
    # 2. محاولة الاستخراج (المرقمة)
    medications_raw = re.findall(r"\d+\.\s+(.*?)(?=\s*\d+\.|$)", meds_section_text)
    
    # =========================================================
    # 🔄 Fallback Mechanism: إذا لم نجد قائمة مرقمة، نفصل بالفواصل
    # =========================================================
    if not medications_raw:
        # التقسيم بالفواصل أو الأسطر الجديدة
        medications_raw = [m.strip() for m in re.split(r'[,\n]', meds_section_text) if m.strip()]

    extracted_meds = []
    
    for med in medications_raw:
        # نمط الفصل (نفس السابق)
        split_pattern = r"(\d+|_+|\bTAB\b|\bCAP\b|\bPO\b|\bBID\b|\bTID\b|\bQ\d+\b|\bPRN\b|\bNEB\b|\bIH\b|\bMG\b|\bMCG\b|\bUNITS\b|\bPUFF\b|\bDOSE\b|\bAMT\b)"
        
        parts = re.split(split_pattern, med, flags=re.IGNORECASE)
        base_name = parts[0].strip()
        
        # تنظيف الرموز
        base_name = base_name.replace('-', ' ').strip(' _.,:')
        
        if base_name and len(base_name) > 1:
            clean_name = re.sub(r'\s+\d+(\.\d+)?$', '', base_name).strip()
            
            # تجاهل الكلمات الشائعة التي ليست أدوية (مثل "None" أو "accurate")
            if clean_name.lower() not in ['none', 'accurate', 'complete', 'list']:
                extracted_meds.append(clean_name)
            
    return extracted_meds

# --- اختبار على النص الجديد (الغير مرقم) ---
text_sample = """
Name: ___. Unit No: ___ Admission Date: ___ Discharge Date: ___ Date of Birth: ___ Sex: F Service: MEDICINE Allergies: Tramadol Attending: ___. Chief Complaint:Abdominal distention, back pain, fever; leukocytosis. Major Surgical or Invasive Procedure:Paracentesis x 3. History of Present Illness:This is a ___ woman with a history of ETOH abuse who presents with abdominal distention, back pain, fever, and elevated white count from Liver Clinic. Ms. ___ was recently admitted to this hospital about 1 week ago for treatment of ascites and work-up of alcoholic hepatitis. At that time she had a diagnostic and therapeutic paracentesis and was treated for a UTI. She was discharged home and instructed to follow-up in Liver Clinic in 1 week. On day of presentation to liver clinic, patient complained of worsening abdominal pain and low-grade fevers at home. Her labwork was also significant for an elevated white count. As such, Ms. ___ was admitted for work-up of fever and white count, and for treatment of recurrent ascites. Past Medical History:--Alcohol abuse--Chronic back pain Social History:___Family History:Breast cancer in mother age ___, No IBD, liver failure. Multiple relatives with alcoholism. Physical Exam:VS: 97.9, 103/73, 86, 18, 96% RA GEN: A/Ox3, pleasant, appropriate, well appearing HEENT: No temporal wasting, JVD not elevated, neck veins fill from above. CV: RRR, No MRG PULM: CTAB but decreased BS in R base. ABD: Distended and tight, diffusely tender to palpation, BS+, + passing flatulence. LIMBS: 2+ edema of the LEs to knee bilaterally ___ pulses 2+ bilaterally NEURO: No asterixis, very mild general tremor. Pertinent Results:Labs at Admission:___ 09:47AM BLOOD WBC-26.2*# RBC-3.86* Hgb-13.0 Hct-43.3 MCV-112* MCH-33.7* MCHC-30.0* RDW-12.7 Plt ______ 09:47AM BLOOD Neuts-88* Bands-1 Lymphs-2* Monos-7 Eos-1 Baso-1 ___ Myelos-0___ 09:20PM BLOOD ______ 09:47AM BLOOD UreaN-8 Creat-0.5 Na-133 K-5.1 Cl-92* HCO3-26 AnGap-20___ 09:47AM BLOOD ALT-45* AST-165* LD(LDH)-345* AlkPhos-200* TotBili-2.0*___ 09:47AM BLOOD Albumin-2.9* Calcium-8.1* Phos-4.0 Mg-2.2___ 09:20PM BLOOD Ethanol-NEG Bnzodzp-NEGLabs at Discharge:___ 07:20AM BLOOD WBC-20.7* RBC-3.03* Hgb-10.3* Hct-32.0* MCV-106* MCH-33.9* MCHC-32.1 RDW-13.7 Plt ______ 07:20AM BLOOD ___ PTT-42.0* ______ 07:20AM BLOOD Glucose-96 UreaN-7 Creat-0.4 Na-125* K-4.4 Cl-90* HCO3-30 AnGap-9___ 07:20AM BLOOD ALT-35 AST-131* LD(___)-265* AlkPhos-184* TotBili-1.9*___ 07:20AM BLOOD Albumin-2.5* Calcium-7.2* Phos-2.6* Mg-2.0Micro Data:___ PERITONEAL FLUID GRAM STAIN- negative; FLUID CULTURE-PENDING; ANAEROBIC CULTURE- negative___ STOOL CLOSTRIDIUM DIFFICILE TOXIN A & B TEST- negative___ URINE URINE CULTURE- negative___ SWAB R/O VANCOMYCIN RESISTANT ENTEROCOCCUS- negative___ STOOL CLOSTRIDIUM DIFFICILE TOXIN A & B TEST- negative___ FLUID RECEIVED IN BLOOD CULTURE BOTTLES Fluid Culture in Bottles- negative___ PERITONEAL FLUID GRAM STAIN-FINAL; FLUID CULTURE-FINAL; ANAEROBIC CULTURE- negative___ BLOOD CULTURE Blood Culture, Routine- negative ___ BLOOD CULTURE Blood Culture, Routine-negative ___ URINE URINE CULTURE-FINAL {GRAM POSITIVE BACTERIA} INPATIENT ___ FLUID RECEIVED IN BLOOD CULTURE BOTTLES negativeImaging Results:CTA (___):1. No evidence of pulmonary embolism. 2. Stable atelectasis at the right lung base. 3. Moderate right and small left pleural effusions, unchanged. CTAP (___):1. Hepatomegaly and large ascites consistent with stated history of liver disease. No evidence of portal venous thrombosis suggesting that the findings on the prior ultrasound may have resulted from extremely slow / undetectable flow. 2. Moderate right and small left pleural effusions, increased on the right with right basilar atelectasis. 3. Replaced right hepatic artery arising from the SMA, otherwise conventional arterial and venous anatomy. Brief Hospital Course:This is a ___ woman with likely alcoholic hepatitis and recurrent ascites who is admitted with low-grade fevers, high white count, and abdominal pain. # ASCITES/ALC HEPATITIS/LEUKOCYTOSIS: Patient with fatty liver and ascites in setting of extensive drinking history and AST/ALT elevation >2. Discriminant function on admission was ~30. Patient had a paracentesis on ___ and 4L was removed; peritoneal fluid was negative for SBP. Diuretics were initially held in the setting of hyponatremia. She was treated supportively with nutrition, brief antibiotics for urinary tract infection (3-days of ceftriaxone), and therapeutic paracenteses x3. Her symptoms, white cell count, and total bilirubin were improving at time of discharge. She will follow-up with Dr. ___ in liver clinic and with her primary care provider, Dr. ___, in two weeks. # HYPONATREMIA: Likely hypovolemic hyponatremia with some component of euvolemic hyponatremia from liver disease. Her spironlactone was held and can be restarted at the discretion of her outpatient liver team, if necessary. Sodium at time of discharge was 125. She has been advised to continue a low sodium diet and free water restriction to ___ liters daily. # ALCOHOLISM: Patient has been trying to cut back recently, but reports daily heavy alcohol intake for the past ___ years; she has had withdrawal symptoms before but no seizures. Shakes and hallucinations. Reports sobriety since prior admission. She will continue outpatient rehab. # URINARY TRACT INFECTION: she was treated with a three-day course of empiric ceftriaxone for concern of UTI.# BACK PAIN/ABDOMINAL PAIN: this was treated in house with lidocaine patches as needed and oxycodone as needed. She has been provided with a short course of Tramadol to take as needed until follow-up with her primary care provider. She understands that this is only a temporary medication and will be discontinued when her acute hepatitis resolves.# Prophylaxis: -DVT ppx with SC heparin -Bowel regimen with lactulose, no PPI -Pain management with oxycodone and lidocaine patch # Communication: Patient # Code: presumed full Medications on Admission:Multivitamin, thiamine, folate, spironolactone 25mg daily, lidocaine patch prn, nicotine patch. Discharge Medications:1. Multivitamin Tablet Sig: One (1) Tablet PO DAILY (Daily). 2. Thiamine HCl 100 mg Tablet Sig: One (1) Tablet PO DAILY (Daily). 3. Folic Acid 1 mg Tablet Sig: One (1) Tablet PO DAILY (Daily). 4. Nicotine 14 mg/24 hr Patch 24 hr Sig: One (1) Patch 24 hr Transdermal DAILY (Daily). 5. Lidocaine 5 %(700 mg/patch) Adhesive Patch, Medicated Sig: One (1) Adhesive Patch, Medicated Topical DAILY (Daily). 6. Tramadol 50 mg Tablet Sig: One (1) Tablet PO every eight (8) hours as needed for pain.Disp:*30 Tablet(s)* Refills:*0* Discharge Disposition:Home Discharge Diagnosis:Primary diagnosis: Alcoholic hepatitis. Discharge Condition:Mental Status: Clear and coherent.Level of Consciousness: Alert and interactive.Activity Status: Ambulatory - Independent. Discharge Instructions:You were admitted to the hospital for alcoholic hepatitis. This is a condition in which your liver becomes inflamed due to excessive alcohol intake. You were also noted to have an elevated white cell count which can sometimes indicate infection. You were treated with a brief course of antibiotics for a urinary tract infection. Otherwise your blood and peritoneal fluid cultures remain negative.We made the following changes to your medications: We stopped your spironolactone because your blood sodium levels were too low.We added Tramadol to take as needed for back pain. Followup Instructions:___

"""

meds = extract_preadmission_medications_v8(text_sample)

print("--- النتائج النهائية للـ API (V8) ---")
for m in meds:
    print(f"'{m}'")

--- النتائج النهائية للـ API (V8) ---
'Multivitamin'
'thiamine'
'folate'
'spironolactone'
'lidocaine patch'
'nicotine patch'


In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 150) # توسيع العرض

# 1. تحميل الملف المثرى
df = pd.read_parquet("../data/processed/GOLD_patient_records_enriched.parquet")

# 2. فحص عينات ناجحة (لنتأكد أننا قصصنا الأدوية فعلاً)
print("✅ Successful Extractions Sample:")
success_samples = df[df['extraction_status'] == 'SUCCESS'].sample(5)
display(success_samples)

# 3. فحص حالات الفشل (لماذا لم نجد القسم؟)
# هذا يساعدنا في تحسين الـ Regex مستقبلاً إذا أردنا
print("\n⚠️ Missing Sections Sample (Check 'full_note_text' to see why):")
missing_samples = df[df['extraction_status'] == 'SECTION_MISSING'].sample(3)
# نعرض أول 1000 حرف فقط لنفهم السياق
missing_samples['full_preview'] = missing_samples['full_note_text'].str[:1000] 
display(missing_samples[['hadm_id', 'full_preview']])

✅ Successful Extractions Sample:


,subject_id,hadm_id,gender,anchor_age,weight,medication_list,dosage_list,unit_list,diagnosis_names,full_note_text,home_meds_raw,extraction_status
2666,10139040,21875071,M,53,194.300000,"[Lisinopril, Simvastatin, Metoprolol Tartrate, Acetaminophen, Sodium Chloride 0.9% Flush, Aspirin, Heparin]","[5, 40, 12.5, 325-650, 3, 325, 5000]","[mg, mg, mg, mg, mL, mg, UNIT]","[Other chest pain, Unspecified essential hypertension]",\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex...,Lisinopril 5mg daily \nSimvastatin 40mg daily \nASA 81mg daily,SUCCESS
45472,12327384,24501982,F,69,152.872881,"[Insulin, Calcium Carbonate, Fenofibrate, Insulin, FoLIC Acid, Losartan Potassium, Polyethylene Glycol, Sodium Chloride 0.9% Flush, Metoprolol Su...","[0, 1500, 96, 0, 1, 50, 17, 3-10, 50, 1500, 12.5, 0.5, 40, 5, 400, 0, 20, 12.5, 0, 15, 1, 100, 250, 500, 100, 12, 100, 400, 1-2, 81, 10, 20, 0, 10...","[UNIT, mg, mg, UNIT, mg, mg, g, mL, mg, mg, mg, mL, mg, mg, mg, UNIT, UNIT, gm, UNIT, g, mg, mg, mL, mg, mg, UNIT, mg, UNIT, PUFF, mg, UNIT, mg, U...","[Type 2 diabetes mellitus with hyperglycemia, Cerebral infarction, unspecified, Hyperosmolality and hypernatremia, Encounter for immunization, Und...",\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Se...,The Preadmission Medication list is accurate and complete.\n1. Albuterol Inhaler ___ PUFF IH Q4H:PRN wheeze or cough \n2. fenofibrate 108 mg Oral ...,SUCCESS
169836,18638156,21323440,F,43,122.091111,"[HYDROmorphone (Dilaudid), Senna, Docusate Sodium, Ondansetron, 0.9% Sodium Chloride (Mini Bag Plus), CeftriaXONE, 0.45% Sodium Chloride, HYDROmor...","[2-4, 1, 100, 4, 100, 1, 1000, 2, 17, 200, 1000, 5000, 5, 20, 5, 1]","[mg, TAB, mg, mg, mL, gm, mL, mg, g, mg, mL, UNIT, mL, mg, mL, PTCH]","[Neoplasm related pain (acute) (chronic), Malignant poorly differentiated neuroendocrine carcinoma, any site, Secondary neuroendocrine tumor of li...",\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: ...,The Preadmission Medication list is accurate and complete.\n1. Prochlorperazine 10 mg PO Q6H:PRN nausea/vomiting \n2. Ondansetron 8 mg PO Q8H:PRN ...,SUCCESS
79313,14058990,23480437,M,60,230.000000,"[Bisacodyl, Bisacodyl, Methylnaltrexone, 5% Dextrose, CeFAZolin, HydrALAZINE, Lidocaine Jelly 2% (Glydo), CARVedilol, Insulin, Omeprazole, HYDROmo...","[10, 10, 12, 100, 2, 25, 1, 25, 0, 20, 4, 4, 100, 20, 8.6, 30, 25, 1, 1, 12.5, 30, 100, 15, 40, 300, 3-10, 30, 3-10, 12, 100, 1000, 40-80]","[mg, mg, mg, mL, g, mg, Appl, mg, UNIT, mg, mg, mg, mL, mg, mg, mg, mg, mg, SPRY, gm, mL, mg, g, mg, mg, mL, mL, mL, mg, mg, mL, mg]","[Other mechanical complication of other internal orthopedic devices, implants and grafts, initial encounter, Pseudarthrosis after fusion or arthro...",\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex:...,Losartan Potassium 100 mg PO DAILY \n CARVedilol 25 mg PO BID \n Chlorthalidone 25 mg PO DAILY \n empagliflozin 10 mg oral DAILY \n Fluticason...,SUCCESS
12395,10645761,24267931,F,60,162.000000,"[Acetaminophen, Famotidine, Influenza Virus Vaccine, Glucose Gel, Acetaminophen-Caff-Butalbital, Iso-Osmotic Sodium Chloride, Famotidine, Senna, S...","[325-650, 20, 0.5, 15, 1-2, 1, 20, 2, 2, 4, 100, 10, 200, 40, 1, 40, 4, 17, 10, 10, 10, 1-5, 40, 100, 10-20, 2, 2, 5-10, 10, 5, 4, 300, 5000, 0, 6...","[mg, mg, mL, g, TAB, BAG, mg, TAB, TAB, mg, mg, mg, mL, mg, mg, mEq, mg, g, mg, mg, mg, mg, mg, mg, mg, TAB, TAB, mg, mg, mg, mg, mg, UNIT, UNIT, ...","[Cerebral aneurysm, nonruptured, Gout, unspecified, Need for prophylactic vaccination and inoculation against influenza, Anxiety state, unspecifie...",\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Se...,"Celexa, Gabapentin, Melatonin, Trazadone, Acamprosate",SUCCESS



⚠️ Missing Sections Sample (Check 'full_note_text' to see why):


,hadm_id,full_preview
148304,26217758,\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: M...
125827,29380749,\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: ...
189813,28804598,\nName: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Se...


In [9]:
import pandas as pd
pd.set_option('display.max_colwidth', 150) # توسيع العرض

# 1. تحميل الملف المثرى
df = pd.read_parquet("../data/processed/GOLD_patient_records_enriched_without_full_text.parquet")

# 2. فحص عينات ناجحة (لنتأكد أننا قصصنا الأدوية فعلاً)
print("✅ Successful Extractions Sample:")
success_samples = df.sample(10)
display(success_samples)

# 3. فحص حالات الفشل (لماذا لم نجد القسم؟)
# هذا يساعدنا في تحسين الـ Regex مستقبلاً إذا أردنا
print("\n⚠️ Missing Sections Sample (Check 'full_note_text' to see why):")
missing_samples = df.sample(10)

✅ Successful Extractions Sample:


,subject_id,hadm_id,gender,anchor_age,weight,medication_list,dosage_list,unit_list,diagnosis_names,home_meds_raw
42560,13480254,28590541,M,85,NaN,"[Oxycodone-Acetaminophen (5mg-325mg), Senna, Metoprolol Tartrate, Ferrous Sulfate, Milk of Magnesia, Warfarin, Sodium Chloride 0.9% Flush, Acetam...","[1-2, 1, 25, 325, 30, 1, 3, 325-650, 40, 1, 20, 20, 40, 81, 1, 40, 20, 1, 100, 10]","[TAB, TAB, mg, mg, mL, dose, mL, mg, mg, dose, mEq, mEq, mg, mg, mg, mg, mg, mg, mg, mg]","[Unspecified pleural effusion, Adult failure to thrive, Coronary atherosclerosis of unspecified type of vessel, native or graft, Atrial fibrillati...",1.Aspirin 81 mg PO DAILY \n2. Metoprolol Tartrate 25 mg PO BID \nHold for HR < 55 or SBP < 90 and call medical provider. \n3. Pravastatin 40 mg PO...
82653,16722175,23474287,M,55,128.321250,"[Insulin, Docusate Sodium, Epoetin Alfa, Insulin, Tacrolimus, Insulin, Sodium Chloride 0.9%, Insulin, Tacrolimus, Heparin Dwell (1000 Units/mL), I...","[0, 100, 3000, 4, 1, 10, 1000, 0, 3, 2000-8000, 0, 650, 2, 30, 2, 4, 14, 1, 1, 8.6, 1000, 4, 20, 1, 30, 12, 3000, 200, 14, 1600, 0, 2000-8000, 4, ...","[UNIT, mg, UNIT, UNIT, mg, UNIT, mL, UNIT, mg, UNIT, UNIT, mg, UNIT, gm, mcg, UNIT, mg, mg, Appl, mg, mg, UNIT, mg, Appl, mg, UNIT, UNIT, mg, UNIT...","[Type 1 diabetes mellitus with hyperglycemia, End stage renal disease, Hypertensive heart and chronic kidney disease with heart failure and with s...",The Preadmission Medication list is accurate and complete.\n1. Acetaminophen 650 mg PO Q6H:PRN Pain - Mild \n2. Labetalol 200 mg PO BID \n3. Docus...
52395,14271697,24873566,F,50,191.133333,"[Lactated Ringers, 5% Dextrose, Morphine Sulfate, amLODIPine, Acetaminophen IV, HydrALAZINE, Heparin, Iso-Osmotic Sodium Chloride, Famotidine, Fam...","[1000, 100, 100, 2.5, 1000, 10, 5000, 1, 20, 20, 5-10, 4, 3-10, 50, 600, 650, 1000, 50, 600]","[mL, mL, mg, mg, mg, mg, UNIT, BAG, mg, mg, mg, mg, mL, mL, mg, mg, mL, mL, mg]","[Morbid (severe) obesity due to excess calories, Fatty (change of) liver, not elsewhere classified, Vitamin D deficiency, unspecified, Obstructive...",The Preadmission Medication list may be inaccurate and requires \nfuther investigation.\n1. MetFORMIN (Glucophage) 500 mg PO BID \n2. amLODIPine 2...
102445,18320272,25948491,F,33,238.750000,"[Influenza Vaccine Quadrivalent, Sodium Chloride 0.9% Flush, Omeprazole, Acetaminophen, Morphine Sulfate, Opium Tincture (morphine 10 mg/mL), Pot...","[0.5, 3, 40, 1000, 5, 10, 1000, 8, 2-4, 0.5, 10, 2, 5000]","[mL, mL, mg, mg, mg, DROP, mL, mg, mg, mg, DROP, mg, UNIT]","[Diarrhea, unspecified, Dehydration, Generalized abdominal pain, Nausea with vomiting, unspecified, Abnormal weight loss, Unspecified asthma, unco...",The Preadmission Medication list may be inaccurate and requires \nfuther investigation.\n1. Albuterol Inhaler 2 PUFF IH Q6H:PRN wheezing \n2. Omep...
63875,15207750,29582380,F,44,147.800000,"[Heparin, Senna, 5% Dextrose, Heparin Sodium, Acetaminophen (Liquid), Sodium Chloride 0.9% Flush, Potassium Chloride (Powder), OxycoDONE (Immedia...","[1300-2600, 8.6, 250, 25,000, 650, 3, 40, 5-10, 20, 10, 0.5, 7.5, 400, 0.5, 25, 1000, 14, 100, 1000, 5-10, 50, 1000]","[UNIT, mg, mL, UNIT, mg, mL, mEq, mg, mg, mg, mL, mg, mg, mg, mg, mL, mg, mg, mg, mg, mg, mg]","[Endometriosis, unspecified, Abnormal uterine and vaginal bleeding, unspecified, Long term (current) use of antithrombotics/antiplatelets, Encount...",Aleve prn
17292,11420248,28942614,F,45,160.120000,"[Heparin, Acetaminophen IV, Levofloxacin, Insulin, Influenza Vaccine Quadrivalent, Insulin, TraMADol, Glucagon, Acetaminophen, Cepacol (Sore Throa...","[5000, 1000, 750, 0, 0.5, 0, 25, 1, 1000, 1, 250, 650, 1, 1, 1, 3-10, 1000, 5, 12.5, 15, 1000, 1, 15]","[UNIT, mg, mg, UNIT, mL, UNIT, mg, mg, mg, LOZ, mg, mg, NEB, VIAL, gm, mL, mg, mL, gm, mg, mg, NEB, g]","[Pneumonia, unspecified organism, Septic pulmonary embolism without acute cor pulmonale, Sepsis, unspecified organism, Essential (primary) hyperte...",The Preadmission Med


⚠️ Missing Sections Sample (Check 'full_note_text' to see why):
